<a href="https://colab.research.google.com/github/sbindal2017-a11y/FineTuned-VisionLLM/blob/main/VLM_Broadcaster_7_INDIAN_Languages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q -U torch transformers accelerate edge-tts moviepy nest_asyncio yt-dlp opencv-python bitsandbytes
!apt install -y ffmpeg

print("✅ Libraries Installed! ready to race.")

In [ ]:

import os
import torch
import json
import asyncio
import edge_tts
import nest_asyncio
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import Audio, display, Video

# Apply nest_asyncio to allow async loops in Colab
nest_asyncio.apply()

# Download a sample F1 clip (5 seconds) if you don't have one
if not os.path.exists("race_clip.mp4"):
    print("⬇️ Downloading sample F1 clip...")
    # Using a generic racing clip for demo purposes
    !yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" --force-overwrites -o "race_clip.mp4" "https://www.youtube.com/watch?v=S-LMSpzlnc0"
    # Trimming it to 10 seconds to keep processing fast
    !ffmpeg -y -i race_clip.mp4 -ss 00:00:30 -t 00:00:10 -c:v libx264 -c:a aac -strict experimental input_video.mp4
    print("✅ Sample video ready: input_video.mp4")
else:
    print("✅ Found 'race_clip.mp4'. Using that.")
    !cp race_clip.mp4 input_video.mp4

In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"🧠 Loading {model_name}...")

# 4-bit Quantization (Crucial for free Colab memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ AI Brain Loaded Successfully!")

# The Voice Map (Microsoft Edge Neural Voices)
VOICE_MAP = {
    "Hindi": "hi-IN-MadhurNeural",      # Male, Authoritative
    "Tamil": "ta-IN-ValluvarNeural",    # Male, Energetic
    "Telugu": "te-IN-MohanNeural",      # Male, Clear
    "Bengali": "bn-IN-BashkarNeural",   # Male, Deep
    "Marathi": "mr-IN-ManoharNeural",   # Male, Sharp
    "English": "en-IN-PrabhatNeural"    # Indian English Accent
}

In [ ]:

import cv2
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- 1. Load the Vision Model (Moondream2) ---
# We use a separate model just for "seeing" the image.
# Moondream is tiny but very good at describing scenes.
print("👀 Loading Vision Model (Moondream2)...")
vision_model_id = "vikhyatk/moondream2"
vision_model = AutoModelForCausalLM.from_pretrained(
    vision_model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="cuda"  # Runs on the same GPU as Gemma
)
vision_tokenizer = AutoTokenizer.from_pretrained(vision_model_id)

# --- 2. Define the 'Extract Frame' Function ---
def extract_frame_at_time(video_path, seconds):
    """
    Opens video, jumps to specific timestamp, reads frame, converts to PIL Image
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_id = int(fps * seconds)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    ret, frame = cap.read()
    cap.release()

    if ret:
        # Convert BGR (OpenCV standard) to RGB (Vision Model standard)
        return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return None

# --- 3. The "Full Stack" Analysis Function ---
def analyze_real_video_segment(video_path, timestamp):

    # A. GET THE IMAGE
    image = extract_frame_at_time(video_path, timestamp)
    if image is None:
        return None

    # B. VISION MODEL: "What do you see?"
    # We ask a specific prompt to get sporty details
    vision_prompt = "Describe this F1 race scene. Mention car colors, crashes, or overtakes."
    enc_image = vision_model.encode_image(image)
    visual_description = vision_model.answer_question(enc_image, vision_prompt, vision_tokenizer)

    print(f"\n[Time: {timestamp}s] 👀 Vision Saw: {visual_description}")

    # C. COMMENTARY MODEL: "Translate to excitement!"
    # Now we pass the *real* description to Gemma
    prompt = f"""<start_of_turn>user
You are an expert Indian sports commentator.
The visual scene is: "{visual_description}"

Write a SHORT, high-energy commentary sentence (max 15 words) for this event in the following languages.
Use Roman Script (English letters) for Hindi, Tamil, Telugu, etc.

Output ONLY a JSON object:
{{
  "Hindi": "...",
  "Tamil": "...",
  "Telugu": "...",
  "English": "..."
}}<end_of_turn>
<start_of_turn>model
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.7)
    raw_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract JSON
    try:
        json_str = raw_text.split("model")[-1].strip()
        json_str = json_str[json_str.find("{"):json_str.rfind("}")+1]
        return json.loads(json_str)
    except:
        return None

# --- TEST IT ---
# We analyze the video at the 5-second mark
print("\n🎬 Analyzing Frame at 00:05...")
script = analyze_real_video_segment("input_video.mp4", timestamp=5)

if script:
    print("\n📝 Generated Script:\n", json.dumps(script, indent=2))
    # Generate Audio for these real results
    audio_files = asyncio.run(generate_audio_files(script))
    print("\n✅ Real audio generated!")

In [ ]:

import time

# Step A: Analyze Video
visual_context = analyze_video_frame("input_video.mp4")

# Step B: Generate Text Script
start_time = time.time()
script = generate_polyglot_script(visual_context)
print(f"\n📝 Script Generated ({time.time() - start_time:.2f}s):\n", json.dumps(script, indent=2))

# Step C: Generate Audio
audio_files = asyncio.run(generate_audio_files(script))
print("\n✅ Audio generation complete!")

In [ ]:

from moviepy.editor import VideoFileClip, AudioFileClip

def create_final_video(lang, video_path="input_video.mp4"):
    audio_path = f"commentary_{lang}.mp3"
    output_path = f"final_output_{lang}.mp4"

    if os.path.exists(audio_path):
        print(f"🎬 Stitching {lang} video...")

        # Load Video
        video = VideoFileClip(video_path)

        # Load Audio & Trim/Loop to match video length
        # (In a real app, you'd match timestamps exactly)
        audio = AudioFileClip(audio_path)

        # Set audio to video
        final_video = video.set_audio(audio)

        # Write file
        final_video.write_videofile(output_path, codec="libx264", audio_codec="aac", logger=None)
        print(f"✅ Saved: {output_path}")
        return output_path
    else:
        print(f"❌ Audio for {lang} not found.")
        return None

# Generate the Hindi Version
final_video_path = create_final_video("Hindi")

# Display it in the Notebook
print("\n👇 WATCH YOUR AI COMMENTARY BELOW 👇")
Video(final_video_path, embed=True, width=600)